<a href="https://colab.research.google.com/github/SilvanaCamboim/Desenvolvimento2025/blob/main/Aula03_Geopandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Preparando o ambiente no Google Drive:

#importando a biblioteca
from google.colab import drive

# Isso irá pedir sua autorização
drive.mount('/content/drive',force_remount=True)

# Agora, seu Drive estará disponível em: /content/drive/My Drive

Mounted at /content/drive


In [ ]:
import geopandas as gpd

# Carrega o shapefile dos municípios
municipios = gpd.read_file("/content/drive/MyDrive/Dados/PR_Municipios_2023.shp")



In [ ]:
# Filtra o município de interesse
mun_selecionado = municipios[municipios["NM_MUN"] == "Castro"]
mun_selecionado.head()



,CD_MUN,NM_MUN,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,CD_REGIAO,NM_REGIAO,CD_CONCURB,NM_CONCURB,AREA_KM2,geometry
69,4104907,Castro,410027,Ponta Grossa,4106,Ponta Grossa,41,Paraná,4,Sul,None,None,2531.503,"POLYGON ((-49.98416 -24.92276, -49.98519 -24.9..."


In [ ]:
# Verifica os vizinhos (usando touches)
vizinhos = municipios[municipios.touches(mun_selecionado.geometry.iloc[0])]
vizinhos.head()

,CD_MUN,NM_MUN,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,CD_REGIAO,NM_REGIAO,CD_CONCURB,NM_CONCURB,AREA_KM2,geometry
58,4104204,Campo Largo,410001,Curitiba,4101,Curitiba,41,Paraná,4,Sul,4106902,Curitiba/PR,1243.551,"POLYGON ((-49.38596 -25.465, -49.38502 -25.465..."
66,4104659,Carambeí,410027,Ponta Grossa,4106,Ponta Grossa,41,Paraná,4,Sul,4119905,Ponta Grossa/PR,649.679,"POLYGON ((-50.11263 -25.00375, -50.11358 -25.0..."
72,4105201,Cerro Azul,410001,Curitiba,4101,Curitiba,41,Paraná,4,Sul,None,None,1341.189,"POLYGON ((-49.27599 -25.0517, -49.27898 -25.04..."
159,4111258,Itaperuçu,410001,Curitiba,4101,Curitiba,41,Paraná,4,Sul,4106902,Curitiba/PR,322.991,"POLYGON ((-49.35287 -25.19769, -49.35254 -25.1..."
268,4119400,Piraí do Sul,410027,Ponta Grossa,4106,Ponta Grossa,41,Paraná,4,Sul,None,None,1345.418,"POLYGON ((-49.95995 -24.61057, -49.96295 -24.6..."


In [ ]:
# Entrada do nome do município
mun = input("Digite o nome de um município: ")

# Filtra o município
selecionado = municipios[municipios['NM_MUN'] == mun]

# Reprojetar para o sistema de coordenadas adequado (29192)
selecionado_proj = selecionado.to_crs(epsg=29192)

# Calcular área em km²
selecionado_proj['area_km2'] = selecionado_proj.geometry.area / 10**6

# Exibir resultado
selecionado_proj[['NM_MUN', 'area_km2']].head()

Digite o nome de um município: Guaratuba


,NM_MUN,area_km2
136,Guaratuba,1327.256679


In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import geopandas as gpd

# 1. Criar Dropdown com nomes e códigos dos municípios
comboMun = Dropdown(
    options=[(row['NM_MUN'], row['CD_MUN']) for _, row in municipios.sort_values('NM_MUN').iterrows()],
    description='Município:'
)

# 2. Definir função de callback
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        codigo = comboMun.value

        # Filtrar o município pelo código
        selecionado = municipios[municipios['CD_MUN'] == codigo]

        # Reprojetar para o sistema de coordenadas em metros (ex: EPSG:29192)
        selecionado_proj = selecionado.to_crs(epsg=29192)

        # Calcular área em km²
        area_km2 = selecionado_proj.geometry.area.iloc[0] / 1e6

        print(f"\nÁrea em km² do município selecionado: {area_km2:.2f}")

# 3. Associar a função ao Dropdown
comboMun.observe(on_change)

# 4. Exibir a interface
display(comboMun)


Dropdown(description='Município:', index=122, options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('A…


Área em km² do município selecionado: 233.98


In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import geopandas as gpd
import folium

# 1. Criar Dropdown
comboMun = Dropdown(
    options=[(row['NM_MUN'], row['CD_MUN']) for _, row in municipios.sort_values('NM_MUN').iterrows()],
    description='Município:'
)

# 2. Callback ao mudar a seleção
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        codigo = comboMun.value

        # Filtrar município selecionado
        selecionado = municipios[municipios['CD_MUN'] == codigo]

        # Calcular área em km² após reprojeção
        selecionado_proj = selecionado.to_crs(epsg=29101)
        area_km2 = selecionado_proj.geometry.area.iloc[0] / 1e6

        print(f"\nÁrea em km² do município selecionado: {area_km2:.2f}")

        # Reprojetar para WGS84 para o folium
        selecionado_wgs84 = selecionado.to_crs(epsg=4326)

        # Criar o mapa
        centro = selecionado_wgs84.geometry.centroid.iloc[0]
        m = folium.Map(location=[centro.y, centro.x], zoom_start=10)

        folium.GeoJson(
            selecionado_wgs84,
            name="Município",
            style_function=lambda x: {"fillColor": "blue", "color": "blue", "weight": 2, "fillOpacity": 0.4}
        ).add_to(m)

        display(m)

# 3. Associar o callback
comboMun.observe(on_change)

# 4. Exibir dropdown
display(comboMun)


Dropdown(description='Município:', index=2, options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('Agu…


Área em km² do município selecionado: 192.78


<ipython-input-7-162d58b349ce>:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centro = selecionado_wgs84.geometry.centroid.iloc[0]


In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import geopandas as gpd
import folium

# 1. Criar Dropdown com nomes e códigos dos municípios
comboMun = Dropdown(
    options=[(row['NM_MUN'], row['CD_MUN']) for _, row in municipios.sort_values('NM_MUN').iterrows()],
    description='Município:'
)

# 2. Função para atualizar o mapa
def atualizar_mapa(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        codigo = comboMun.value

        # Município selecionado
        selecionado = municipios[municipios['CD_MUN'] == codigo]

        # Vizinhos (que "tocam" o polígono selecionado)
        vizinhos = municipios[
            (municipios['CD_MUN'] != codigo) &
            (municipios.touches(selecionado.geometry.iloc[0]))
        ]

        # Reprojetar para WGS84 para uso no folium
        selecionado_wgs84 = selecionado.to_crs(epsg=4326)
        vizinhos_wgs84 = vizinhos.to_crs(epsg=4326)

        # Criar mapa centralizado no município
        centro = selecionado_wgs84.geometry.centroid.iloc[0]
        m = folium.Map(location=[centro.y, centro.x], zoom_start=10)

        # Adicionar município selecionado em azul
        folium.GeoJson(
            selecionado_wgs84,
            name="Selecionado",
            style_function=lambda x: {"fillColor": "blue", "color": "blue", "weight": 2, "fillOpacity": 0.4}
        ).add_to(m)

        # Adicionar vizinhos em vermelho
        folium.GeoJson(
            vizinhos_wgs84,
            name="Vizinhos",
            style_function=lambda x: {"fillColor": "red", "color": "red", "weight": 1.5, "fillOpacity": 0.3}
        ).add_to(m)

        folium.LayerControl().add_to(m)
        display(m)

# 3. Conectar a função ao Dropdown
comboMun.observe(atualizar_mapa)

# 4. Mostrar a interface
display(comboMun)


Dropdown(description='Município:', index=247, options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('A…

<ipython-input-8-22abae33e9c9>:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centro = selecionado_wgs84.geometry.centroid.iloc[0]
